# LOAN DEFAULT RISK PREDICTION MACHINE LEARNING MODEL 

### LOGISTIC REGRESSION AND XGBOOST





### DATA COLLECTION

#### Import important libraries

In [ ]:
import pandas as pd
import numpy as np
import shap

#### Import data visualisation libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#### Import datasets from the csv file

In [ ]:
app=pd.read_csv("Downloads/Loan default risk prediction data/HC_application_train.csv")
prev_app=pd.read_csv("Downloads/Loan default risk prediction data/HC_previous_application.csv")
bureau=pd.read_csv('Downloads/Loan default risk prediction data/HC_bureau.csv')

### EXPLORATORY DATA ANALYSIS

In [ ]:
app.head()

In [ ]:
prev_app.head()

In [ ]:
bureau.head()

In [ ]:
bureau['SK_ID_CURR'].duplicated().value_counts()

In [ ]:
app.describe()

### FEATURE ENGENEERING

##### Debt to income ratio

In [ ]:
app['DTI_RATIO']=app['AMT_ANNUITY']/app['AMT_INCOME_TOTAL']

In [ ]:
app["DTI_RATIO"].describe()

##### Credit to income ratio

In [ ]:
app['CREDIT_TO_INCOME_RATIO']=app['AMT_CREDIT']/app['AMT_INCOME_TOTAL']

In [ ]:
app['CREDIT_TO_INCOME_RATIO'].describe()

In [ ]:
app[app['CREDIT_TO_INCOME_RATIO']==app['CREDIT_TO_INCOME_RATIO'].max()]

#### Age of credit history from the bureau

In [ ]:
bureau_agg=bureau.groupby('SK_ID_CURR').agg(BUREAU_DAYS_CREDIT_MIN=('DAYS_CREDIT','min'),
                                           BUREAU_DAYS_CREDIT_MAX=('DAYS_CREDIT','max'),
                                           BUREAU_CREDIT_ACTIVE= ('CREDIT_ACTIVE', lambda x: (x=='Active').sum()),
                                           TOTAL_BUREAU_CREDIT_DAY_OVERDUE=('CREDIT_DAY_OVERDUE', lambda x: (x>0).sum()))

In [ ]:
app=app.merge(bureau_agg,on='SK_ID_CURR',how='left')

#### Number of the previous loans

In [ ]:
prev_app['NAME_CONTRACT_STATUS'].value_counts()

In [ ]:
prev_app_agg=prev_app.groupby('SK_ID_CURR').agg(NUMBER_OF_PAST_APPS=('SK_ID_PREV',lambda x: (x).count()),
                                                PREVIOUS_REFUSED_RATIO=('NAME_CONTRACT_STATUS', lambda x:(x=='Refused').mean()))

In [ ]:
prev_app_agg.describe()

In [ ]:
app=app.merge(prev_app_agg,on='SK_ID_CURR',how='left')

In [ ]:
app.head()

#### Employment period

In [ ]:
app[app['DAYS_EMPLOYED']>=0]['DAYS_EMPLOYED'].value_counts()

##### There are days recorded as 365243 which is an anomaly and it should be handled

In [ ]:
app['DAYS_EMPLOYED']=app['DAYS_EMPLOYED'].replace({365243: np.nan})

In [ ]:
app['YEARS_EMPLOYED']=app['DAYS_EMPLOYED']/-365.25

In [ ]:
app['YEARS_EMPLOYED'].describe()

#### Applicant's age

In [ ]:
app['AGE']=app['DAYS_BIRTH']/-365.25

In [ ]:
app['AGE'].describe()

#### EXTERNAL SOURCES SCORES

In [ ]:
app[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].describe()

In [ ]:
app['EXT_SOURCE_MEAN']=app[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)

### DATA CLEANING AND TRANSFORMATION

In [ ]:
sns.heatmap(app.isnull())

##### Missing values flag

Create two new columns "EXT_SOURCE_1_MISSING AND AMT_ANNUITY_MISSING" and fill values 1 for the missing and 0 for the ones with values

In [ ]:
for col in ['EXT_SOURCE_1','AMT_ANNUITY']:
   app[f'{col}_MISSING']=app[col].isnull().astype(int)

In [ ]:
app.shape

#### Encoding Applicant's gender

In [ ]:
app['CODE_GENDER']=app['CODE_GENDER'].map({'F':0,'M':1,'XNA':2})

In [ ]:
app['CODE_GENDER'].head()

#### Fill the remaining NaN

In [ ]:
app=app.fillna(app.median(numeric_only=True))

In [ ]:
app[app['SK_ID_CURR']==app['SK_ID_CURR'].isnull()]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score,classification_report,confusion_matrix

### Feature selection 

In [ ]:
features=['DTI_RATIO','CREDIT_TO_INCOME_RATIO','BUREAU_DAYS_CREDIT_MIN','BUREAU_DAYS_CREDIT_MAX',
          'BUREAU_CREDIT_ACTIVE','TOTAL_BUREAU_CREDIT_DAY_OVERDUE','NUMBER_OF_PAST_APPS',
          'PREVIOUS_REFUSED_RATIO','YEARS_EMPLOYED','EXT_SOURCE_MEAN','AGE','CODE_GENDER','CNT_CHILDREN']

In [ ]:
X=app[features]
y=app['TARGET']

In [ ]:
sns.heatmap(X.isnull())

#### Splitting data into train and test data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

#### Model training

In [ ]:
LogReg=LogisticRegression(max_iter=1000,class_weight='balanced')
LogReg.fit(X_train,y_train)

In [ ]:
y_pred_prob=LogReg.predict_proba(X_test)[:,1]

In [ ]:
y_pred_prob

#### Roc_Auc and average_Precision_score

In [ ]:
print("Roc_Auc:",roc_auc_score(y_test,y_pred_prob))
print("average_precision_score:",average_precision_score(y_test,y_pred_prob))

#### Threshold selection using cost minimising function

In [ ]:
from sklearn.metrics import precision_recall_curve

In [ ]:
precision, recall, thresholds= precision_recall_curve(y_test,y_pred_prob)
cost_fn=0.8
cost_fp=0.2

costs=[]
approval_rate=[]
bad_rate=[]

for threshold in thresholds:
    preds=(y_pred_prob>=threshold).astype(int)
    fp=((y_test==0)&(preds==1)).sum()
    fn=((y_test==1)&(preds==0)).sum()
    tp=((y_test==1)&(preds==1)).sum()
    tn=((y_test==0)&(preds==0)).sum()
    costs.append(fp*cost_fp+fn*cost_fn)
    approval_rate.append((preds==0).mean())
    bad_rate.append(fn/(tn+fn+1e-9))

best_threshold=thresholds[np.argmin(costs)]
plt.figure(figsize=(10,5))
plt.plot(thresholds,costs,label="total Cost",linewidth=3)
plt.plot(thresholds, np.array(approval_rate)*100,label='Approval Rate %',linewidth=3)
plt.plot(thresholds, np.array(bad_rate)*100,label="Bad Rate % in Approved", linewidth=3)

best_threshold=thresholds[np.argmin(costs)]
plt.axvline(best_threshold,color="blue",ls="--",label=f"Optimal threshold={best_threshold:,.3f}")
plt.xlabel("Threshold")
plt.ylabel("Value")
plt.title("Threshold selection for unsecured loan")
plt.legend()
plt.grid(True)
plt.show()

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Approval rate at best threshold : {np.mean((y_pred_prob<best_threshold))*100:.2f}%")
print(f"Bad rate at best threshold : {np.mean(y_test[y_pred_prob<best_threshold])*100:.2f}%")

#### Calibrate the probabilities for Logistic Regression Model

In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

calbrated = CalibratedClassifierCV(LogReg,cv=5,method="sigmoid")
calbrated.fit(X_train,y_train)
y_pred_prob_cal=calbrated.predict_proba(X_test)[:,1]
print("Brier score before:", brier_score_loss(y_test,y_pred_prob))
print("Brier score after:", brier_score_loss(y_test,y_pred_prob_cal))

In [ ]:
from sklearn.calibration import calibration_curve
precision, recall, thresholds= precision_recall_curve(y_test,y_pred_prob_cal)
cost_fn=0.8
cost_fp=0.2

costs=[]
approval_rate=[]
bad_rate=[]

for threshold in thresholds:
    preds=(y_pred_prob_cal>=threshold).astype(int)
    fp=((y_test==0)&(preds==1)).sum()
    fn=((y_test==1)&(preds==0)).sum()
    tp=((y_test==1)&(preds==1)).sum()
    tn=((y_test==0)&(preds==0)).sum()
    costs.append(fp*cost_fp+fn*cost_fn)
    approval_rate.append((preds==0).mean())
    bad_rate.append(fn/(tn+fn + 1e-9))
    
best_threshold=thresholds[np.argmin(costs)]
    

fig, ax = plt.subplots(1,2,figsize=(10,5))
prob_true,prob_pred=calibration_curve(y_test,y_pred_prob,n_bins=10)
ax[0].plot(prob_pred,prob_true,marker="o")
ax[0].plot([0,1],[0,1],ls="--")
ax[0].set_title("Before Calibration")
ax[0].set_xlabel("Predicted Prob")
ax[0].set_ylabel("Actual Prob")

prob_true_cal,prob_pred_cal=calibration_curve(y_test,y_pred_prob_cal,n_bins=10)
ax[1].plot(prob_pred_cal,prob_true_cal,marker="o")
ax[1].plot([0,1],[0,1],ls="--")
ax[1].set_title("After calibration")
plt.show()

#### Threshold after calibration method

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(thresholds,costs,label="total Cost",linewidth=3)
plt.plot(thresholds, np.array(approval_rate)*100,label='Approval Rate %',linewidth=3)
plt.plot(thresholds, np.array(bad_rate)*100,label="Bad Rate % in Approved", linewidth=3)

best_threshold=thresholds[np.argmin(costs)]
plt.axvline(best_threshold,color="blue",ls="--",label=f"Optimal threshold={best_threshold:,.3f}")
plt.xlabel("Threshold")
plt.ylabel("Value")
plt.title("Threshold selection for unsecured loan after calibration")
plt.legend()
plt.grid(True)
plt.show()

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Approval rate at best threshold : {np.mean((y_pred_prob_cal<best_threshold))*100:.2f}%")
print(f"Bad rate at best threshold : {np.mean(y_test[y_pred_prob_cal<best_threshold])*100:.2f}%")

#### Results with a Threshold calculated from a cost_based method

In [ ]:
treshold= 0.187

In [ ]:
LogReg_pred_labels=(y_pred_prob>treshold).astype(int)

In [ ]:
LogReg_pred_labels

In [ ]:
print(classification_report(y_test,LogReg_pred_labels))
print(confusion_matrix(y_test,LogReg_pred_labels))

In [ ]:
LogReg_tp=((y_test==1) & (LogReg_pred_labels==1)).sum()
LogReg_fp=((y_test==0) & (LogReg_pred_labels==1)).sum()
LogReg_fn=((y_test==1) & (LogReg_pred_labels==0)).sum()
LogReg_tn=((y_test==0) & (LogReg_pred_labels==0)).sum()

In [ ]:
print('LogReg true positives:',LogReg_tp)
print('LogReg false positives:',LogReg_fp)
print('LogReg false negatives:',LogReg_fn)
print('LogReg true negative:',LogReg_tn)

### Business Impact from Logistic Regression model

In [ ]:
average_loan=25250
loss_given_difault = 0.8
review_cost = 50

In [ ]:
gross_loss_prevented=LogReg_tp*average_loan*loss_given_difault 
ops_cost=(LogReg_tp+LogReg_fp)*review_cost
Net_benefit=gross_loss_prevented-ops_cost

In [ ]:
print(f"Model flags:{(LogReg_tp+LogReg_fp)} aps")
print(f"Caught defaults: {LogReg_tp} | Missed defaults: {LogReg_fn}")
print(f"Amount lost on rejecting good borrower:R {LogReg_fp*average_loan*(1-loss_given_difault):,.2f}|Amount lost through Missed defaults:R {LogReg_fn*average_loan*loss_given_difault:,.2f}")
print(f"Expected Loss: R{(LogReg_fp*average_loan*(1-loss_given_difault)+LogReg_fn*average_loan*loss_given_difault):,.2f}")
print(f"Gross_loss_prevented: R{gross_loss_prevented:,.2f}")
print(f"Operations cost: R{ops_cost:,.2f}")
print(f"Net benefit : R{Net_benefit:,.2f}")
print(f"Recall:{LogReg_tp/(LogReg_tp+LogReg_fn):,.2f} | Precision:{LogReg_tp/(LogReg_tp+LogReg_fp):,.2f}")

#### Better performance

In [ ]:
from xgboost import XGBClassifier 
from sklearn.calibration  import CalibratedClassifierCV

In [ ]:
xgb=XGBClassifier(n_estimators=300,
                  max_depth=4, 
                  learning_rate=0.02,
                  subsample=0.8,
                  colsample_bytree=0.8,
                  scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                  eval_metric="auc",random_state=1)

In [ ]:
xgb.fit(X_train,y_train)

In [ ]:
y_xgb_pred_prob=xgb.predict_proba(X_test)[:,1]

#### Roc_Auc and average_Precision_score

In [ ]:
print("Roc_Auc:",roc_auc_score(y_test,y_xgb_pred_prob))
print("average_precision_score:",average_precision_score(y_test,y_xgb_pred_prob))

#### Calibrated probabilities for xgboost

In [ ]:
calibrated_xgb=CalibratedClassifierCV(xgb,method="sigmoid",cv=5)
calibrated_xgb.fit(X_train,y_train)

In [ ]:
y_xgb_prob_cal=calibrated_xgb.predict_proba(X_test)[:,1]

In [ ]:
print("Brier score before:", brier_score_loss(y_test,y_xgb_pred_prob))
print("Brier score after:", brier_score_loss(y_test,y_xgb_prob_cal))

In [ ]:
from sklearn.calibration import calibration_curve
precision, recall, thresholds= precision_recall_curve(y_test,y_xgb_prob_cal)
cost_fn=0.8
cost_fp=0.2

costs=[]
approval_rate=[]
bad_rate=[]

for threshold in thresholds:
    preds=(y_xgb_prob_cal>=threshold).astype(int)
    fp=((y_test==0)&(preds==1)).sum()
    fn=((y_test==1)&(preds==0)).sum()
    tp=((y_test==1)&(preds==1)).sum()
    tn=((y_test==0)&(preds==0)).sum()
    costs.append(fp*cost_fp+fn*cost_fn)
    approval_rate.append((preds==0).mean())
    bad_rate.append(fn/(tn+fn + 1e-9))
    
best_threshold=thresholds[np.argmin(costs)]
    

fig, ax = plt.subplots(1,2,figsize=(10,5))
prob_true,prob_pred=calibration_curve(y_test,y_pred_prob,n_bins=10)
ax[0].plot(prob_pred,prob_true,marker="o")
ax[0].plot([0,1],[0,1],ls="--")
ax[0].set_title("Before Calibration")
ax[0].set_xlabel("Predicted Prob")
ax[0].set_ylabel("Actual Prob")

prob_true_cal,prob_pred_cal=calibration_curve(y_test,y_pred_prob_cal,n_bins=10)
ax[1].plot(prob_pred_cal,prob_true_cal,marker="o")
ax[1].plot([0,1],[0,1],ls="--")
ax[1].set_title("After calibration")
ax[1].set_xlabel("Predicted Prob")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(thresholds,costs,label="total Cost",linewidth=3)
plt.plot(thresholds, np.array(approval_rate)*100,label='Approval Rate %',linewidth=3)
plt.plot(thresholds, np.array(bad_rate)*100,label="Bad Rate % in Approved", linewidth=3)

best_threshold=thresholds[np.argmin(costs)]
plt.axvline(best_threshold,color="blue",ls="--",label=f"Optimal threshold={best_threshold:,.3f}")
plt.xlabel("Threshold")
plt.ylabel("Value")
plt.title("Threshold selection for unsecured loan")
plt.legend()
plt.grid(True)
plt.show()

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Approval rate at best threshold : {np.mean((y_pred_prob_cal<best_threshold))*100:.2f}%")
print(f"Bad rate at best threshold : {np.mean(y_test[y_pred_prob_cal<best_threshold])*100:.2f}%")

In [ ]:
for name,model in [('XGBOOST',xgb)]:
    xgb_pred=model.predict_proba(X_test)[:,1]
print(f"{name} ROC_AUC : {roc_auc_score(y_test,xgb_pred):,.2f}")
print(f"{name} PR_AUC:{average_precision_score(y_test,xgb_pred):,.2f}")

##### XGB predicted labels with a threshold= 0.22

In [ ]:
treshold=0.202

In [ ]:
xgb_pred_labels=(xgb_pred>treshold).astype(int)

In [ ]:
print(classification_report(y_test,xgb_pred_labels))
print(confusion_matrix(y_test,xgb_pred_labels))

In [ ]:
xgb_tp=((y_test==1)& (xgb_pred_labels==1)).sum()
xgb_fp=((y_test==0)& (xgb_pred_labels==1)).sum()
xgb_fn=((y_test==1)& (xgb_pred_labels==0)).sum()
xgb_tn=((y_test==0)& (xgb_pred_labels==0)).sum()

In [ ]:
print('xgb true positives:',xgb_tp)
print('xgb false positives:',xgb_fp)
print('xgb false negatives:',xgb_fn)
print('xgb true negative:',xgb_tn)

### Business Impact from xgboost

In [ ]:
gross_loss_prevented=xgb_tp*average_loan*loss_given_difault 
ops_cost=(xgb_tp+xgb_fp)*review_cost
xgb_Net_benefit=gross_loss_prevented-ops_cost

In [ ]:
print(f"Model flags:{(xgb_tp+xgb_fp)} aps")
print(f"Caught defaults: {xgb_tp} | Missed defaults: {xgb_fn}")
print(f"Amount lost on rejecting good borrower:R {xgb_fp*average_loan*(1-loss_given_difault):,.2f}|Amount lost through Missed defaults:R {xgb_fn*average_loan*loss_given_difault:,.2f}")
print(f"Expected Loss: R{(xgb_fp*average_loan*(1-loss_given_difault)+xgb_fn*average_loan*loss_given_difault):,.2f}")
print(f"Gross_loss_prevented: R{gross_loss_prevented:,.2f}")
print(f"Operations cost: R{ops_cost:,.2f}")
print(f"xgb_Net benefit : R{xgb_Net_benefit:,.2f}")
print(f"Recall:{xgb_tp/(xgb_tp+xgb_fn):,.2f} | Precision:{xgb_tp/(xgb_tp+xgb_fp):,.2f}")

#### XGBOOST has performed poorly compared to Logistic regression model, hence going back to Logistic Regression model is the best decision

#### SHAP plot for explainability

In [ ]:
explainer=shap.Explainer(LogReg,X_train)
shap_values=explainer(X_test.iloc[:1000])
shap.plots.beeswarm(shap_values)

#### SHAP waterfall plot for 1 loan applicant 

In [ ]:
for i in [0,100]:
    shap.waterfall_plot(shap.Explanation(
        values=explainer.shap_values(X_test.iloc[i]),
        base_values=explainer.expected_value,
        data=X_test.iloc[i],
        feature_names=X_test.columns))

#### Saving a Logistic Rregression model

In [ ]:
import joblib
joblib.dump(LogReg,'model.pkl')